In [ ]:
import re
import pandas as pd
from pathlib import Path

IN_CSV  = Path(r"E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\result\preprocessing_result\sirah_simple_clean.csv")        # hasil preprocess
OUT_CSV = Path(r"E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\preprocess_result\csv_result\sirah_chunks_final.csv") # hasil chunking

df = pd.read_csv(IN_CSV, sep=";", encoding="utf-8-sig").fillna("")
df.columns = df.columns.astype(str).str.replace("\ufeff", "", regex=False).str.strip()

print("Rows dokumen:", len(df))
df.head(3)

Rows dokumen: 389


,judul_bab,judul_sub_bab,halaman,teks_clean
0,POSISI BANGSA ARAB DAN KAUMNYA,UNLABELED SECTION,34,Pada hakikatnya istilah Sirah Nabawiyah merupa...
1,POSISI BANGSA ARAB DAN KAUMNYA,Posisi Bangsa Arab,34-35,"Menurut bahasa, Arab artinya padang pasir, tan..."
2,POSISI BANGSA ARAB DAN KAUMNYA,Kaum-kaum Bangsa Arab,35-42,Ditilik dari silsilah keturunan dan cikal-baka...


In [2]:
_SENT_SPLIT = re.compile(r"(?<=[.!?])\s+")

def split_sentences(text: str):
    text = (text or "").strip()
    if not text:
        return []
    t = re.sub(r"\s+", " ", text).strip()
    return [s.strip() for s in _SENT_SPLIT.split(t) if s.strip()]

def chunk_sentences(sentences, max_chars=1500, overlap_sents=1):
    """
    Gabungkan kalimat menjadi chunk <= max_chars.
    overlap_sents: jumlah kalimat terakhir chunk sebelumnya yang diulang pada chunk berikutnya.
    """
    chunks = []
    buf = []

    def buf_text_len(b):
        return len(" ".join(b)) if b else 0

    for s in sentences:
        s = s.strip()
        if not s:
            continue

        # kalau menambah kalimat ini melebihi max_chars, tutup chunk dulu
        if buf and (buf_text_len(buf) + 1 + len(s)) > max_chars:
            chunks.append(" ".join(buf).strip())

            # overlap: bawa 1 kalimat terakhir
            if overlap_sents > 0:
                buf = buf[-overlap_sents:]
            else:
                buf = []

        buf.append(s)

    if buf:
        chunks.append(" ".join(buf).strip())

    return [c for c in chunks if c]


In [3]:
# Pastikan kolom teks yang dipakai benar
if "teks_clean" not in df.columns:
    raise ValueError(f"Kolom 'teks_clean' tidak ditemukan. Kolom yang ada: {df.columns.tolist()}")

rows = []
total_empty = 0

for doc_id, r in df.iterrows():
    text = (r["teks_clean"] or "").strip()
    if not text:
        total_empty += 1
        continue

    sents = split_sentences(text)

    # fallback: kalau OCR minim tanda baca, jadikan 1 "kalimat"
    if not sents:
        sents = [re.sub(r"\s+", " ", text).strip()]

    chunks = chunk_sentences(sents, max_chars=1500, overlap_sents=1)

    for i, ch in enumerate(chunks, start=1):
        rows.append({
            "chunk_id": f"{doc_id:06d}-{i:03d}",
            "doc_id": doc_id,
            "chunk_index": i,
            "judul_bab": r["judul_bab"],
            "judul_sub_bab": r["judul_sub_bab"],
            "halaman": r["halaman"],    # tetap range (mis: 45-47)
            "teks_chunk": ch
        })

df_chunks = pd.DataFrame(rows)
print("Dokumen kosong:", total_empty)
print("Total chunks:", len(df_chunks))
df_chunks.head(5)


Dokumen kosong: 0
Total chunks: 1099


,chunk_id,doc_id,chunk_index,judul_bab,judul_sub_bab,halaman,teks_chunk
0,000000-001,0,1,POSISI BANGSA ARAB DAN KAUMNYA,UNLABELED SECTION,34,Pada hakikatnya istilah Sirah Nabawiyah merupa...
1,000001-001,1,1,POSISI BANGSA ARAB DAN KAUMNYA,Posisi Bangsa Arab,34-35,"Menurut bahasa, Arab artinya padang pasir, tan..."
2,000001-002,1,2,POSISI BANGSA ARAB DAN KAUMNYA,Posisi Bangsa Arab,34-35,Sekalipun begitu mereka tetap hidup berdamping...
3,000002-001,2,1,POSISI BANGSA ARAB DAN KAUMNYA,Kaum-kaum Bangsa Arab,35-42,Ditilik dari silsilah keturunan dan cikal-baka...
4,000002-002,2,2,POSISI BANGSA ARAB DAN KAUMNYA,Kaum-kaum Bangsa Arab,35-42,"Thayyi', Madzhij, Kindah, Lakham, Judzam, Uzd,..."


In [4]:
df_chunks.to_csv(OUT_CSV, index=False, sep=";", encoding="utf-8-sig")
print("Saved:", OUT_CSV.resolve())

Saved: E:\2_Kehidupan-Kuliah\10_tugas-akhir\repo-TA\TA_preprocess\TA_sirah\data\preprocess_result\csv_result\sirah_chunks_final.csv


In [5]:
df_chunks["len"] = df_chunks["teks_chunk"].str.len()
df_chunks["len"].describe()
df_chunks.sample(10)[["chunk_id","halaman","teks_chunk"]]

,chunk_id,halaman,teks_chunk
816,000269-001,483-484,Setelah benteng Az-Zubair dapat direbut dan di...
997,000357-005,577-590,"""Akulah Ka'b bin Zuhair,"" kata Ka'b Seorang An..."
169,000049-012,121-129,"Beliau bersabda, ""Sabarlah wahai keluarga Yasi..."
70,000016-002,73-74,"bin Yatsribi, bin Yahzan, bin Yalhan, bin Ar'a..."
284,000080-003,191-197,Di sana beliau melihat Idris. Beliau mengucapk...
36,000008-002,53-54,Sehingga adakalanya jika seorang pemimpin murk...
97,000024-002,87,Kami juga bisa mendapatkan tanda itu di dalam ...
390,000112-001,254-255,Kemudian orang-orang Quraisy mengirim pasukan ...
879,000301-002,515-517,Mereka juga diperintahkan untuk menyiarkan kab...
33,000007-009,48-53,"Setelah Qushay meninggal dunia, kewenangan ini..."
